#### **Zadanie LSTM - generator przepisów**

In [2]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader

# Hiperparametry
SEQ_LENGTH = 40
BATCH_SIZE = 32
HIDDEN_DIM = 256
NUM_LAYERS = 2
EMBEDDING_DIM = 64
EPOCHS = 50
LEARNING_RATE = 0.002
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Używane urządzenie: {DEVICE}")

Używane urządzenie: cpu


In [4]:
TEXT_DATA = """
Składniki: 2 jajka, 1 szklanka mąki, pół szklanki mleka, szczypta soli.
Przygotowanie: Wymieszaj wszystkie składniki na gładką masę. Smaż na rozgrzanej patelni z obu stron na złoty kolor. Podawaj z dżemem.
[RECIPE_END]
Składniki: 500g mięsa mielonego, 1 cebula, puszka pomidorów, makaron spaghetti.
Przygotowanie: Cebulę posiekaj i podsmaż. Dodaj mięso i smaż przez 10 minut. Wlej pomidory, dopraw solą i pieprzem. Gotuj sos przez 20 minut. Podawaj z ugotowanym makaronem.
[RECIPE_END]
Składniki: 3 ziemniaki, 1 marchewka, 1 pietruszka, bulion, koperek.
Przygotowanie: Warzywa obierz i pokrój w kostkę. Wrzuć do gorącego bulionu i gotuj do miękkości. Na koniec posyp świeżym koperkiem.
[RECIPE_END]
Składniki: 100g masła, 200g czekolady, 3 jajka, pół szklanki cukru, łyżka mąki.
Przygotowanie: Masło i czekoladę rozpuść w kąpieli wodnej. Jajka ubij z cukrem. Połącz masę czekoladową z jajeczną, dodaj mąkę. Piecz w 180 stopniach przez 25 minut.
[RECIPE_END]
"""

chars = tuple(set(TEXT_DATA))
int2char = dict(enumerate(chars))
char2int = {ch: ii for ii, ch in int2char.items()}
VOCAB_SIZE = len(chars)

encoded_text = np.array([char2int[ch] for ch in TEXT_DATA])
print(f"Vocab Size: {VOCAB_SIZE}")
print(f"Przykładowe kodowanie: '{TEXT_DATA[0:10]}' -> {encoded_text[0:10]}")

Vocab Size: 55
Przykładowe kodowanie: '
Składniki' -> [14 27 48 16 53  3 52  5 48  5]


In [6]:
#przygotowanie danych
class RecipeDataset(Dataset):
    def __init__(self, encoded_text, seq_length):
        self.encoded_text = encoded_text
        self.seq_length = seq_length

    def __len__(self):
        return len(self.encoded_text) - self.seq_length

    def __getitem__(self, idx):
        x = self.encoded_text[idx : idx + self.seq_length]
        y = self.encoded_text[idx + 1 : idx + self.seq_length + 1]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

dataset = RecipeDataset(encoded_text, SEQ_LENGTH)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

In [8]:
#Model
class RecipeGeneratorLSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hid_dim, n_layers, batch_first=True)
        self.fc = nn.Linear(hid_dim, vocab_size)

    def forward(self, x, hidden=None):
        embedded = self.embedding(x)
        out, hidden = self.lstm(embedded, hidden)
        logits = self.fc(out) 
        return logits, hidden

def generate_text(model, start_str="Składniki:", length=150, temperature=0.8):
    model.eval()
    chars_input = [char2int.get(c, 0) for c in start_str]
    input_tensor = torch.tensor(chars_input).unsqueeze(0).to(DEVICE)
    
    generated_text = start_str
    hidden = None 
    
    with torch.no_grad():
        for _ in range(length):
            output, hidden = model(input_tensor, hidden)
            logits = output[0, -1, :] / temperature
            probs = torch.softmax(logits, dim=0)
            next_char_idx = torch.multinomial(probs, num_samples=1).item()
            
            generated_text += int2char[next_char_idx]
            input_tensor = torch.tensor([[next_char_idx]]).to(DEVICE)
            
    return generated_text

In [13]:
#trening
from IPython.display import clear_output

model = RecipeGeneratorLSTM(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    
    for x_batch, y_batch in dataloader:
        x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
        
        optimizer.zero_grad()
        output, _ = model(x_batch)
        loss = criterion(output.view(-1, VOCAB_SIZE), y_batch.view(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
        optimizer.step()
        
        total_loss += loss.item()
        
    if epoch % 5 == 0 or epoch == 1:
        #clear_output(wait=True) # czysci ekran!
        avg_loss = total_loss / len(dataloader)
        print(f"=== Epoka: {epoch}/{EPOCHS} | Strata (Loss): {avg_loss:.4f} ===")
        sample_text = generate_text(model, start_str="Składniki: ", length=150, temperature=0.7)
        print(f"{sample_text}\n{'-'*50}")

=== Epoka: 1/50 | Strata (Loss): 3.3473 ===
Składniki: mla Wyuio ras.ckmrkaie dsoaneksu keodij, c 2zez ooia poąeic zlko  ossz g otrtreuo iu, goigbubą ma0 sokra klua s zwna  osoaajd iz omuiaa peereu oo so i
--------------------------------------------------
=== Epoka: 5/50 | Strata (Loss): 0.2056 ===
Składniki: 100g masła, 200g czekolady, 3 jajka, pół szklanki cukru, łyżka mąki.
Przygotowanie: Warzywa obierz i pokrój w kostkę. Wrzuć do gorącego bulionu i gotu
--------------------------------------------------
=== Epoka: 10/50 | Strata (Loss): 0.1254 ===
Składniki: 100g masła, 200g czekolady, 3 jajka, pół szklanki cukru, łyżka mąki.
Przygotowanie: Masło i czekoladę rozpuść w kąpieli wodnej. Jajka ubij z cukrem. P
--------------------------------------------------
=== Epoka: 15/50 | Strata (Loss): 0.1110 ===
Składniki: 2 jajka, 1 szklanka mąki, pół szklanki mleka, szczypta soli.
Przygotowanie: Wymieszaj wszystkie składniki na gładką masę. Smaż na rozgrzanej patelni z
---------------------

In [25]:
#Playground
# Zmień tekst startowy i temperaturę!
# Temperatura > 1 = więcej szaleństwa (model dodaje wymyślone litery)
# Temperatura < 1 = model bezpieczny i powtarzalny
wygenerowany_przepis = generate_text(model, start_str="Składniki: ", length=200, temperature=0.5)
print(wygenerowany_przepis)

Składniki: 3 ziemniaki, 1 marchewka, 1 pietruszka, bulion, koperek.
Przygotowanie: Warzywa obierz i pokrój w kostkę. Wrzuć do gorącego bulionu i gotuj do miękkości. Na koniec posyp świeżym koperkiem.
[RECIPE_END
